In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
sns.set_theme(style="whitegrid", context="talk")

bundle_dir = Path("visualizations/report_bundle")
if not (bundle_dir / "FINAL_runs_lenient_single_1000").exists() and (bundle_dir / "experiment_bundle").exists():
    bundle_dir = bundle_dir / "experiment_bundle"


In [ ]:
SHORT_NAME_MAP = {
    "FINAL_runs_lenient_single_1000": "baseline",
    "runs_lenient_wider2_kfold5_drop_empty": "less_slices",
    "runs_lenient_wider2_kfold5_drop_empty_bs1": "less_slices_bs1",
    "runs_lenient_wider2_kfold5_zspacing_only": "z_crop",
    "runs_lenient_wider2_kfold5_zspacing_xycrop": "xyz_crop",
    "runs_lenient_wider2_kfold5_zspacing_dicece": "z_crop_dicece",
    "runs_lenient_wider2_kfold5_zspacing_dice_sched": "z_crop_sched",
}

INCLUDED_RUNS = [
    "baseline",
    "z_crop",
    "xyz_crop",
    "z_crop_dicece",
    "z_crop_sched",
]


def short_name(folder_name):
    return SHORT_NAME_MAP.get(folder_name, folder_name)


experiment_dirs = sorted(
    [
        p for p in bundle_dir.iterdir()
        if p.is_dir() and short_name(p.name) in INCLUDED_RUNS
    ]
)

print("Bundle directory:", bundle_dir.resolve())
print("Included experiment folders:")
for p in experiment_dirs:
    print(" -", p.name, "->", short_name(p.name))


In [ ]:
SEGMENTATION_METRICS = [
    "loss_train", "loss_val", "loss_test",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
]

CLASSIFIER_METRICS = [
    "macro_f1_train", "macro_f1_val", "macro_f1_test",
    "min_disease_recall_train", "min_disease_recall_val", "min_disease_recall_test",
]

ALL_METRICS = SEGMENTATION_METRICS + CLASSIFIER_METRICS
PHASE_ORDER = ["ED", "ES"]
STRUCTURE_ORDER = ["LV", "MYO", "RV"]
DISEASE_ORDER = ["DCM", "HCM", "MINF", "NOR", "RV"]
SPLIT_ORDER = ["train", "val", "test"]


def mean_numeric(series):
    return pd.to_numeric(series, errors="coerce").mean()


def add_run_meta(df, exp_dir):
    if df is None or df.empty:
        return df

    out = df.copy()
    out.insert(0, "folder", exp_dir.name)
    out.insert(0, "run", short_name(exp_dir.name))
    return out


def build_phase_summary(patient_df):
    if patient_df is None or patient_df.empty:
        return pd.DataFrame(columns=["split", "phase", "dice", "hd95"])

    patient_phase_df = (
        patient_df
        .assign(
            dice=pd.to_numeric(patient_df["dice"], errors="coerce"),
            hd95=pd.to_numeric(patient_df["hd95"], errors="coerce"),
        )
        .groupby(["split", "ID", "phase"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    phase_df = (
        patient_phase_df
        .groupby(["split", "phase"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    phase_df["split"] = pd.Categorical(phase_df["split"], categories=SPLIT_ORDER, ordered=True)
    phase_df["phase"] = pd.Categorical(phase_df["phase"], categories=PHASE_ORDER, ordered=True)
    return phase_df.sort_values(["split", "phase"]).reset_index(drop=True)


def build_disease_summary(patient_df):
    if patient_df is None or patient_df.empty:
        return pd.DataFrame(columns=["split", "Disease", "dice", "hd95"])

    patient_disease_df = (
        patient_df
        .assign(
            dice=pd.to_numeric(patient_df["dice"], errors="coerce"),
            hd95=pd.to_numeric(patient_df["hd95"], errors="coerce"),
        )
        .groupby(["split", "ID", "Disease"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    disease_df = (
        patient_disease_df
        .groupby(["split", "Disease"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    disease_df["split"] = pd.Categorical(disease_df["split"], categories=SPLIT_ORDER, ordered=True)
    disease_df["Disease"] = pd.Categorical(disease_df["Disease"], categories=DISEASE_ORDER, ordered=True)
    return disease_df.sort_values(["split", "Disease"]).reset_index(drop=True)


def build_structure_summary(patient_df):
    if patient_df is None or patient_df.empty:
        return pd.DataFrame(columns=["split", "structure", "dice", "hd95"])

    patient_structure_df = (
        patient_df
        .assign(
            dice=pd.to_numeric(patient_df["dice"], errors="coerce"),
            hd95=pd.to_numeric(patient_df["hd95"], errors="coerce"),
        )
        .groupby(["split", "ID", "structure"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    structure_df = (
        patient_structure_df
        .groupby(["split", "structure"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )

    structure_df["split"] = pd.Categorical(structure_df["split"], categories=SPLIT_ORDER, ordered=True)
    structure_df["structure"] = pd.Categorical(structure_df["structure"], categories=STRUCTURE_ORDER, ordered=True)
    return structure_df.sort_values(["split", "structure"]).reset_index(drop=True)


def load_patient_metrics(patient_path):
    if not patient_path.exists():
        return None
    return pd.read_csv(patient_path)


def load_kfold_run(exp_dir):
    row = {
        "run": short_name(exp_dir.name),
        "folder": exp_dir.name,
    }

    fold_results_path = exp_dir / "fold_results.csv"
    if fold_results_path.exists():
        fold_df = pd.read_csv(fold_results_path)
        for metric in ALL_METRICS:
            if metric in fold_df.columns:
                row[metric] = mean_numeric(fold_df[metric])

    patient_df = load_patient_metrics(exp_dir / "evaluation_final" / "patient_metrics_long.csv")
    phase_df = add_run_meta(build_phase_summary(patient_df), exp_dir)
    disease_df = add_run_meta(build_disease_summary(patient_df), exp_dir)
    structure_df = add_run_meta(build_structure_summary(patient_df), exp_dir)
    return row, phase_df, disease_df, structure_df


def load_single_run(exp_dir):
    row = {
        "run": short_name(exp_dir.name),
        "folder": exp_dir.name,
    }

    seg_path = exp_dir / "evaluation_final_single" / "split_segmentation_summary.csv"
    if seg_path.exists():
        seg_df = pd.read_csv(seg_path)
        for split in SPLIT_ORDER:
            split_df = seg_df[seg_df["split"] == split]
            if not split_df.empty:
                row[f"dice_{split}"] = mean_numeric(split_df["dice_mean"])
                row[f"hd95_{split}"] = mean_numeric(split_df["hd95_mean"])

    clf_path = exp_dir / "evaluation_final_single" / "classifier_summary.csv"
    if clf_path.exists():
        clf_df = pd.read_csv(clf_path)
        for split in SPLIT_ORDER:
            split_df = clf_df[clf_df["split"] == split]
            if split_df.empty:
                continue
            if "macro_f1" in split_df.columns:
                row[f"macro_f1_{split}"] = mean_numeric(split_df["macro_f1"])
            if "min_disease_recall" in split_df.columns:
                row[f"min_disease_recall_{split}"] = mean_numeric(split_df["min_disease_recall"])

    patient_df = load_patient_metrics(exp_dir / "evaluation_final_single" / "patient_metrics_long.csv")
    phase_df = add_run_meta(build_phase_summary(patient_df), exp_dir)
    disease_df = add_run_meta(build_disease_summary(patient_df), exp_dir)
    structure_df = add_run_meta(build_structure_summary(patient_df), exp_dir)
    return row, phase_df, disease_df, structure_df


def load_experiment(exp_dir):
    if (exp_dir / "fold_results.csv").exists():
        return load_kfold_run(exp_dir)
    if (exp_dir / "evaluation_final_single" / "split_segmentation_summary.csv").exists():
        return load_single_run(exp_dir)

    row = {
        "run": short_name(exp_dir.name),
        "folder": exp_dir.name,
    }
    empty_phase = pd.DataFrame(columns=["run", "folder", "split", "phase", "dice", "hd95"])
    empty_disease = pd.DataFrame(columns=["run", "folder", "split", "Disease", "dice", "hd95"])
    empty_structure = pd.DataFrame(columns=["run", "folder", "split", "structure", "dice", "hd95"])
    return row, empty_phase, empty_disease, empty_structure


def build_phase_wide(phase_long_df):
    if phase_long_df.empty:
        return pd.DataFrame(columns=["run", "folder"])

    phase_wide_df = phase_long_df.pivot_table(
        index=["run", "folder"],
        columns=["phase", "split"],
        values=["dice", "hd95"],
        observed=False,
    )

    ordered_cols = []
    for metric in ["dice", "hd95"]:
        for phase in PHASE_ORDER:
            for split in SPLIT_ORDER:
                col = (metric, phase, split)
                if col in phase_wide_df.columns:
                    ordered_cols.append(col)

    phase_wide_df = phase_wide_df[ordered_cols]
    phase_wide_df.columns = [f"{metric}_{phase}_{split}" for metric, phase, split in phase_wide_df.columns]
    return phase_wide_df.reset_index()


def build_disease_wide(disease_long_df):
    if disease_long_df.empty:
        return pd.DataFrame(columns=["run", "folder"])

    disease_wide_df = disease_long_df.pivot_table(
        index=["run", "folder"],
        columns=["Disease", "split"],
        values=["dice", "hd95"],
        observed=False,
    )

    ordered_cols = []
    for metric in ["dice", "hd95"]:
        for disease in DISEASE_ORDER:
            for split in SPLIT_ORDER:
                col = (metric, disease, split)
                if col in disease_wide_df.columns:
                    ordered_cols.append(col)

    disease_wide_df = disease_wide_df[ordered_cols]
    disease_wide_df.columns = [f"{metric}_{disease}_{split}" for metric, disease, split in disease_wide_df.columns]
    return disease_wide_df.reset_index()


def build_structure_wide(structure_long_df):
    if structure_long_df.empty:
        return pd.DataFrame(columns=["run", "folder"])

    structure_wide_df = structure_long_df.pivot_table(
        index=["run", "folder"],
        columns=["structure", "split"],
        values=["dice", "hd95"],
        observed=False,
    )

    ordered_cols = []
    for metric in ["dice", "hd95"]:
        for structure in STRUCTURE_ORDER:
            for split in SPLIT_ORDER:
                col = (metric, structure, split)
                if col in structure_wide_df.columns:
                    ordered_cols.append(col)

    structure_wide_df = structure_wide_df[ordered_cols]
    structure_wide_df.columns = [f"{metric}_{structure}_{split}" for metric, structure, split in structure_wide_df.columns]
    return structure_wide_df.reset_index()


In [ ]:
rows = []
phase_parts = []
disease_parts = []
structure_parts = []

for exp_dir in experiment_dirs:
    row, phase_df, disease_df, structure_df = load_experiment(exp_dir)
    rows.append(row)
    if phase_df is not None and not phase_df.empty:
        phase_parts.append(phase_df)
    if disease_df is not None and not disease_df.empty:
        disease_parts.append(disease_df)
    if structure_df is not None and not structure_df.empty:
        structure_parts.append(structure_df)

main_results_df = pd.DataFrame(rows)

base_cols = [
    "run",
    "folder",
    "loss_train", "loss_val", "loss_test",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
    "macro_f1_train", "macro_f1_val", "macro_f1_test",
    "min_disease_recall_train", "min_disease_recall_val", "min_disease_recall_test",
]

main_cols = [col for col in base_cols if col in main_results_df.columns]
main_results_df = main_results_df[main_cols].copy()

numeric_cols = [col for col in main_results_df.columns if col not in ["run", "folder"]]
main_results_df[numeric_cols] = main_results_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

main_results_df["run"] = pd.Categorical(main_results_df["run"], categories=INCLUDED_RUNS, ordered=True)
main_results_df = main_results_df.sort_values("run").reset_index(drop=True)

phase_long_df = pd.concat(phase_parts, ignore_index=True) if phase_parts else pd.DataFrame(
    columns=["run", "folder", "split", "phase", "dice", "hd95"]
)
phase_wide_df = build_phase_wide(phase_long_df)

disease_long_df = pd.concat(disease_parts, ignore_index=True) if disease_parts else pd.DataFrame(
    columns=["run", "folder", "split", "Disease", "dice", "hd95"]
)
disease_wide_df = build_disease_wide(disease_long_df)

structure_long_df = pd.concat(structure_parts, ignore_index=True) if structure_parts else pd.DataFrame(
    columns=["run", "folder", "split", "structure", "dice", "hd95"]
)
structure_wide_df = build_structure_wide(structure_long_df)

run_index_df = main_results_df[["run", "folder"]].copy()
phase_wide_df = run_index_df.merge(phase_wide_df, on=["run", "folder"], how="left")
disease_wide_df = run_index_df.merge(disease_wide_df, on=["run", "folder"], how="left")
structure_wide_df = run_index_df.merge(structure_wide_df, on=["run", "folder"], how="left")

for df in [phase_wide_df, disease_wide_df, structure_wide_df]:
    value_cols = [col for col in df.columns if col not in ["run", "folder"]]
    if value_cols:
        df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")

print("Built main results:", main_results_df.shape)
print("Built phase summary:", phase_wide_df.shape)
print("Built disease summary:", disease_wide_df.shape)
print("Built structure summary:", structure_wide_df.shape)


In [ ]:
selection_cols = [
    "run",
    "dice_train", "dice_val", "dice_test",
    "hd95_train", "hd95_val", "hd95_test",
    "macro_f1_train", "macro_f1_val", "macro_f1_test",
    "min_disease_recall_train", "min_disease_recall_val", "min_disease_recall_test",
]

selection_cols = [col for col in selection_cols if col in main_results_df.columns]
display(main_results_df[selection_cols].round(4))

main_results_df.to_csv(bundle_dir / "summary_results.csv", index=False)
print("Saved:", bundle_dir / "summary_results.csv")


In [ ]:
display(phase_wide_df.round(4))

phase_wide_df.to_csv(bundle_dir / "summary_phase_results.csv", index=False)
print("Saved:", bundle_dir / "summary_phase_results.csv")


In [ ]:
display(disease_wide_df.round(4))

disease_wide_df.to_csv(bundle_dir / "summary_disease_results.csv", index=False)
print("Saved:", bundle_dir / "summary_disease_results.csv")


In [ ]:
RUN_ORDER = [
    "baseline",
    "z_crop",
    "xyz_crop",
    "z_crop_dicece",
    "z_crop_sched",
]

BEST_RUN = "z_crop"

RUN_COLORS = {
    "baseline": "#222222",
    "z_crop": "#31a354",
    "xyz_crop": "#fdae6b",
    "z_crop_dicece": "#756bb1",
    "z_crop_sched": "#dd3497",
}

SPLIT_COLORS = {
    "train": "#4C78A8",
    "val": "#F58518",
    "test": "#54A24B",
}


def apply_run_order(df):
    out = df.copy()
    present = out["run"].astype(str).tolist()
    ordered = [run for run in RUN_ORDER if run in present]
    ordered += [run for run in present if run not in ordered]
    out["run"] = pd.Categorical(out["run"], categories=ordered, ordered=True)
    return out.sort_values("run").reset_index(drop=True)


def display_run_label(run_name):
    run_name = str(run_name)
    if run_name == BEST_RUN:
        return f"{run_name} (best)"
    return run_name


def add_run_labels(df):
    out = df.copy()
    out["run_label"] = out["run"].astype(str).map(display_run_label)
    return out


def highlight_best_ticklabels(ax):
    for tick in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        if "(best)" in tick.get_text():
            tick.set_fontweight("heavy")


In [ ]:
test_overall_df = apply_run_order(
    main_results_df[["run", "dice_test", "hd95_test", "macro_f1_test"]].copy()
)
test_overall_df = add_run_labels(test_overall_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
metric_specs = [
    ("dice_test", "Test Dice"),
    ("hd95_test", "Test HD95"),
    ("macro_f1_test", "Test Macro-F1"),
]

for ax, (metric, title) in zip(axes, metric_specs):
    plot_df = test_overall_df.dropna(subset=[metric]).copy()
    colors = [RUN_COLORS.get(str(run), "#4C78A8") for run in plot_df["run"]]
    ax.bar(plot_df["run_label"], pd.to_numeric(plot_df[metric]), color=colors)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    highlight_best_ticklabels(ax)
    for idx, value in enumerate(pd.to_numeric(plot_df[metric])):
        ax.text(idx, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Test Overall Metrics", fontsize=18, y=1.04)
plt.show()


In [ ]:
dice_phase_plot_df = apply_run_order(
    main_results_df[["run", "dice_test"]].merge(
        phase_wide_df[["run", "dice_ED_test", "dice_ES_test"]],
        on="run",
        how="left",
    )
)
dice_phase_plot_df = add_run_labels(dice_phase_plot_df)

dice_phase_long_df = dice_phase_plot_df.melt(
    id_vars=["run", "run_label"],
    value_vars=["dice_test", "dice_ED_test", "dice_ES_test"],
    var_name="metric",
    value_name="value",
).dropna(subset=["value"])

dice_phase_long_df["metric"] = dice_phase_long_df["metric"].map({
    "dice_test": "Overall",
    "dice_ED_test": "ED",
    "dice_ES_test": "ES",
})

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
sns.barplot(
    data=dice_phase_long_df,
    x="run_label",
    y="value",
    hue="metric",
    palette={"Overall": "#4C78A8", "ED": "#F58518", "ES": "#54A24B"},
    ax=ax,
)
ax.set_title("Test Dice Overall And By Phase")
ax.set_xlabel("")
ax.set_ylabel("Dice")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="")
highlight_best_ticklabels(ax)
plt.show()


In [ ]:
hd95_phase_plot_df = apply_run_order(
    main_results_df[["run", "hd95_test"]].merge(
        phase_wide_df[["run", "hd95_ED_test", "hd95_ES_test"]],
        on="run",
        how="left",
    )
)
hd95_phase_plot_df = add_run_labels(hd95_phase_plot_df)

hd95_phase_long_df = hd95_phase_plot_df.melt(
    id_vars=["run", "run_label"],
    value_vars=["hd95_test", "hd95_ED_test", "hd95_ES_test"],
    var_name="metric",
    value_name="value",
).dropna(subset=["value"])

hd95_phase_long_df["metric"] = hd95_phase_long_df["metric"].map({
    "hd95_test": "Overall",
    "hd95_ED_test": "ED",
    "hd95_ES_test": "ES",
})

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
sns.barplot(
    data=hd95_phase_long_df,
    x="run_label",
    y="value",
    hue="metric",
    palette={"Overall": "#4C78A8", "ED": "#F58518", "ES": "#54A24B"},
    ax=ax,
)
ax.set_title("Test HD95 Overall And By Phase")
ax.set_xlabel("")
ax.set_ylabel("HD95")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="")
highlight_best_ticklabels(ax)
plt.show()


In [ ]:
best_run = "z_crop"
best_overall_df = main_results_df[main_results_df["run"].astype(str) == best_run].copy()

if best_overall_df.empty:
    print(f"Run not found: {best_run}")
else:
    best_overall_row = best_overall_df.iloc[0]
    best_phase_df = phase_wide_df[phase_wide_df["run"].astype(str) == best_run].copy()
    best_phase_row = best_phase_df.iloc[0] if not best_phase_df.empty else None

    split_long_records = []
    for split in SPLIT_ORDER:
        split_long_records.append({
            "split": split,
            "panel": "Dice",
            "series": "Overall",
            "value": pd.to_numeric(best_overall_row.get(f"dice_{split}"), errors="coerce"),
        })
        split_long_records.append({
            "split": split,
            "panel": "HD95",
            "series": "Overall",
            "value": pd.to_numeric(best_overall_row.get(f"hd95_{split}"), errors="coerce"),
        })
        split_long_records.append({
            "split": split,
            "panel": "Macro-F1",
            "series": "Overall",
            "value": pd.to_numeric(best_overall_row.get(f"macro_f1_{split}"), errors="coerce"),
        })

        if best_phase_row is not None:
            split_long_records.append({
                "split": split,
                "panel": "Dice",
                "series": "ED",
                "value": pd.to_numeric(best_phase_row.get(f"dice_ED_{split}"), errors="coerce"),
            })
            split_long_records.append({
                "split": split,
                "panel": "Dice",
                "series": "ES",
                "value": pd.to_numeric(best_phase_row.get(f"dice_ES_{split}"), errors="coerce"),
            })
            split_long_records.append({
                "split": split,
                "panel": "HD95",
                "series": "ED",
                "value": pd.to_numeric(best_phase_row.get(f"hd95_ED_{split}"), errors="coerce"),
            })
            split_long_records.append({
                "split": split,
                "panel": "HD95",
                "series": "ES",
                "value": pd.to_numeric(best_phase_row.get(f"hd95_ES_{split}"), errors="coerce"),
            })

    split_long_df = pd.DataFrame(split_long_records).dropna(subset=["value"]).copy()
    panel_order = ["Dice", "HD95", "Macro-F1"]
    series_palette = {"Overall": "#4C78A8", "ED": "#F58518", "ES": "#54A24B"}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

    for ax, panel_name in zip(axes, panel_order):
        plot_df = split_long_df[split_long_df["panel"] == panel_name].copy()
        sns.barplot(
            data=plot_df,
            x="split",
            y="value",
            hue="series",
            palette=series_palette,
            ax=ax,
        )
        ax.set_title(panel_name)
        ax.set_xlabel("")
        if panel_name == "Macro-F1":
            ax.legend(title="")
        else:
            ax.legend(title="")

    fig.suptitle("Z Crop (Best) Across Train Val Test", fontsize=18, y=1.05)
    plt.show()


In [ ]:
disease_test_plot_df = apply_run_order(
    disease_wide_df[[
        "run",
        "dice_DCM_test", "dice_HCM_test", "dice_MINF_test", "dice_NOR_test", "dice_RV_test",
        "hd95_DCM_test", "hd95_HCM_test", "hd95_MINF_test", "hd95_NOR_test", "hd95_RV_test",
    ]].copy()
)

dice_heat_df = disease_test_plot_df[[
    "run",
    "dice_DCM_test", "dice_HCM_test", "dice_MINF_test", "dice_NOR_test", "dice_RV_test",
]].copy()
dice_heat_df = dice_heat_df.set_index("run")
dice_heat_df.index = [display_run_label(run) for run in dice_heat_df.index]
dice_heat_df.columns = ["DCM", "HCM", "MINF", "NOR", "RV"]

hd95_heat_df = disease_test_plot_df[[
    "run",
    "hd95_DCM_test", "hd95_HCM_test", "hd95_MINF_test", "hd95_NOR_test", "hd95_RV_test",
]].copy()
hd95_heat_df = hd95_heat_df.set_index("run")
hd95_heat_df.index = [display_run_label(run) for run in hd95_heat_df.index]
hd95_heat_df.columns = ["DCM", "HCM", "MINF", "NOR", "RV"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

sns.heatmap(
    dice_heat_df,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8},
    ax=axes[0],
)
axes[0].set_title("Per-Disease Test Dice")
axes[0].set_xlabel("")
axes[0].set_ylabel("")
highlight_best_ticklabels(axes[0])

sns.heatmap(
    hd95_heat_df,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd_r",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8},
    yticklabels=False,
    ax=axes[1],
)
axes[1].set_title("Per-Disease Test HD95")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
highlight_best_ticklabels(axes[1])

plt.show()


In [ ]:
structure_test_plot_df = apply_run_order(
    structure_wide_df[[
        "run",
        "dice_LV_test", "dice_MYO_test", "dice_RV_test",
        "hd95_LV_test", "hd95_MYO_test", "hd95_RV_test",
    ]].copy()
)

dice_heat_df = structure_test_plot_df[[
    "run",
    "dice_LV_test", "dice_MYO_test", "dice_RV_test",
]].copy()
dice_heat_df = dice_heat_df.set_index("run")
dice_heat_df.index = [display_run_label(run) for run in dice_heat_df.index]
dice_heat_df.columns = ["LV", "MYO", "RV"]

hd95_heat_df = structure_test_plot_df[[
    "run",
    "hd95_LV_test", "hd95_MYO_test", "hd95_RV_test",
]].copy()
hd95_heat_df = hd95_heat_df.set_index("run")
hd95_heat_df.index = [display_run_label(run) for run in hd95_heat_df.index]
hd95_heat_df.columns = ["LV", "MYO", "RV"]

fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

sns.heatmap(
    dice_heat_df,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8},
    ax=axes[0],
)
axes[0].set_title("Per-Structure Test Dice")
axes[0].set_xlabel("")
axes[0].set_ylabel("")
highlight_best_ticklabels(axes[0])

sns.heatmap(
    hd95_heat_df,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd_r",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8},
    yticklabels=False,
    ax=axes[1],
)
axes[1].set_title("Per-Structure Test HD95")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
highlight_best_ticklabels(axes[1])

plt.show()


In [ ]:
phase_structure_parts = []

for exp_dir in experiment_dirs:
    patient_path = exp_dir / "evaluation_final" / "patient_metrics_long.csv"
    if not patient_path.exists():
        patient_path = exp_dir / "evaluation_final_single" / "patient_metrics_long.csv"
    if not patient_path.exists():
        continue

    patient_df = pd.read_csv(patient_path)
    if patient_df.empty:
        continue

    detail_df = (
        patient_df
        .assign(
            run=short_name(exp_dir.name),
            dice=pd.to_numeric(patient_df["dice"], errors="coerce"),
            hd95=pd.to_numeric(patient_df["hd95"], errors="coerce"),
        )
        .groupby(["run", "split", "phase", "structure"], observed=False)
        .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
        .reset_index()
    )
    phase_structure_parts.append(detail_df)

if not phase_structure_parts:
    print("No phase-structure patient metrics found.")
else:
    phase_structure_df = pd.concat(phase_structure_parts, ignore_index=True)
    phase_structure_df = phase_structure_df[phase_structure_df["split"] == "test"].copy()
    phase_structure_df["run"] = pd.Categorical(
        phase_structure_df["run"], categories=INCLUDED_RUNS, ordered=True
    )
    phase_structure_df["phase"] = pd.Categorical(
        phase_structure_df["phase"], categories=PHASE_ORDER, ordered=True
    )
    phase_structure_df["structure"] = pd.Categorical(
        phase_structure_df["structure"], categories=STRUCTURE_ORDER, ordered=True
    )
    phase_structure_df = phase_structure_df.sort_values(["phase", "run", "structure"]).reset_index(drop=True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    for row_idx, phase in enumerate(PHASE_ORDER):
        phase_df = phase_structure_df[phase_structure_df["phase"] == phase].copy()

        dice_heat_df = (
            phase_df
            .pivot(index="run", columns="structure", values="dice")
            .reindex(INCLUDED_RUNS)
        )
        dice_heat_df.index = [display_run_label(run) for run in dice_heat_df.index]
        dice_heat_df = dice_heat_df.reindex(columns=STRUCTURE_ORDER)

        hd95_heat_df = (
            phase_df
            .pivot(index="run", columns="structure", values="hd95")
            .reindex(INCLUDED_RUNS)
        )
        hd95_heat_df.index = [display_run_label(run) for run in hd95_heat_df.index]
        hd95_heat_df = hd95_heat_df.reindex(columns=STRUCTURE_ORDER)

        sns.heatmap(
            dice_heat_df,
            annot=True,
            fmt=".3f",
            cmap="YlGnBu",
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"shrink": 0.8},
            ax=axes[row_idx, 0],
        )
        axes[row_idx, 0].set_title(f"{phase} Dice")
        axes[row_idx, 0].set_xlabel("")
        axes[row_idx, 0].set_ylabel("")
        highlight_best_ticklabels(axes[row_idx, 0])

        sns.heatmap(
            hd95_heat_df,
            annot=True,
            fmt=".2f",
            cmap="YlOrRd_r",
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"shrink": 0.8},
            yticklabels=False,
            ax=axes[row_idx, 1],
        )
        axes[row_idx, 1].set_title(f"{phase} HD95")
        axes[row_idx, 1].set_xlabel("")
        axes[row_idx, 1].set_ylabel("")
        highlight_best_ticklabels(axes[row_idx, 1])


    plt.show()
   


In [ ]:
def build_phase_structure_fold_extreme_df(dice_agg="min", hd95_agg="max"):
    fold_parts = []

    for exp_dir in experiment_dirs:
        patient_path = exp_dir / "evaluation_final" / "patient_metrics_long.csv"
        if not patient_path.exists():
            patient_path = exp_dir / "evaluation_final_single" / "patient_metrics_long.csv"
        if not patient_path.exists():
            continue

        patient_df = pd.read_csv(patient_path)
        if patient_df.empty:
            continue

        if "fold" not in patient_df.columns:
            patient_df = patient_df.copy()
            patient_df["fold"] = 1

        fold_df = (
            patient_df
            .assign(
                run=short_name(exp_dir.name),
                dice=pd.to_numeric(patient_df["dice"], errors="coerce"),
                hd95=pd.to_numeric(patient_df["hd95"], errors="coerce"),
            )
            .groupby(["run", "fold", "split", "phase", "structure"], observed=False)
            .agg(dice=("dice", "mean"), hd95=("hd95", "mean"))
            .reset_index()
        )
        fold_parts.append(fold_df)

    if not fold_parts:
        return pd.DataFrame()

    fold_metric_df = pd.concat(fold_parts, ignore_index=True)
    fold_metric_df = fold_metric_df[fold_metric_df["split"] == "test"].copy()

    extreme_df = (
        fold_metric_df
        .groupby(["run", "phase", "structure"], observed=False)
        .agg(dice=("dice", dice_agg), hd95=("hd95", hd95_agg))
        .reset_index()
    )

    extreme_df["run"] = pd.Categorical(extreme_df["run"], categories=INCLUDED_RUNS, ordered=True)
    extreme_df["phase"] = pd.Categorical(extreme_df["phase"], categories=PHASE_ORDER, ordered=True)
    extreme_df["structure"] = pd.Categorical(extreme_df["structure"], categories=STRUCTURE_ORDER, ordered=True)
    return extreme_df.sort_values(["phase", "run", "structure"]).reset_index(drop=True)


def plot_phase_structure_extreme_heatmaps(extreme_df, title):
    if extreme_df.empty:
        print("No fold-level phase-structure metrics found.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    for row_idx, phase in enumerate(PHASE_ORDER):
        phase_df = extreme_df[extreme_df["phase"] == phase].copy()

        dice_heat_df = (
            phase_df
            .pivot(index="run", columns="structure", values="dice")
            .reindex(INCLUDED_RUNS)
        )
        dice_heat_df.index = [display_run_label(run) for run in dice_heat_df.index]
        dice_heat_df = dice_heat_df.reindex(columns=STRUCTURE_ORDER)

        hd95_heat_df = (
            phase_df
            .pivot(index="run", columns="structure", values="hd95")
            .reindex(INCLUDED_RUNS)
        )
        hd95_heat_df.index = [display_run_label(run) for run in hd95_heat_df.index]
        hd95_heat_df = hd95_heat_df.reindex(columns=STRUCTURE_ORDER)

        sns.heatmap(
            dice_heat_df,
            annot=True,
            fmt=".3f",
            cmap="YlGnBu",
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"shrink": 0.8},
            ax=axes[row_idx, 0],
        )
        axes[row_idx, 0].set_title(f"{phase} Dice")
        axes[row_idx, 0].set_xlabel("")
        axes[row_idx, 0].set_ylabel("")
        highlight_best_ticklabels(axes[row_idx, 0])

        sns.heatmap(
            hd95_heat_df,
            annot=True,
            fmt=".2f",
            cmap="YlOrRd_r",
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"shrink": 0.8},
            yticklabels=False,
            ax=axes[row_idx, 1],
        )
        axes[row_idx, 1].set_title(f"{phase} HD95")
        axes[row_idx, 1].set_xlabel("")
        axes[row_idx, 1].set_ylabel("")
        highlight_best_ticklabels(axes[row_idx, 1])

    fig.suptitle(title, fontsize=18, y=1.02)
    plt.show()


worst_fold_phase_structure_df = build_phase_structure_fold_extreme_df(dice_agg="min", hd95_agg="max")
plot_phase_structure_extreme_heatmaps(worst_fold_phase_structure_df, "Worst Fold Per-Structure Test Scores By Phase")


In [ ]:
best_fold_phase_structure_df = build_phase_structure_fold_extreme_df(dice_agg="max", hd95_agg="min")
plot_phase_structure_extreme_heatmaps(best_fold_phase_structure_df, "Best Fold Per-Structure Test Scores By Phase")
